In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model and Loss

In [ ]:

import math
import numpy as np
import torch
from torch import nn

class ChannelAwareNAOMI(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=64, decoder_hidden=128, highest=16,
                 rnn_layers=1, dropout=0.0, normalize=True):
        super().__init__()
        assert highest > 0 and highest & (highest - 1) == 0
        self.data_dim = data_dim
        self.hidden_dim = hidden_dim
        self.highest = highest
        self.normalize = normalize
        self.encoder = nn.GRU(data_dim*2, hidden_dim, num_layers=rnn_layers,
                              bidirectional=True, batch_first=True,
                              dropout=dropout if rnn_layers > 1 else 0.)
        self.decoders = nn.ModuleDict()
        for step in (2**p for p in range(int(math.log2(highest))+1)):
            self.decoders[str(step)] = nn.Sequential(
                nn.Linear(hidden_dim*2 + data_dim*2 + 2, decoder_hidden),
                nn.ReLU(), nn.Linear(decoder_hidden, data_dim))
        # CPU schedule cache, not checkpoint parameters. One entry per unique mask.
        self._schedule_cache = {}

    def _norm(self, x, m):
        valid = m.bool()
        count = valid.sum(dim=1, keepdim=True)
        safe_count = count.clamp_min(1)
        observed = torch.where(valid, x, torch.zeros_like(x))
        mean = observed.sum(dim=1, keepdim=True) / safe_count
        variance = torch.where(valid, (x-mean).square(), torch.zeros_like(x)).sum(dim=1, keepdim=True) / safe_count
        sd = (variance + 1e-6).sqrt()
        empty = count == 0
        mean = torch.where(empty, torch.zeros_like(mean), mean)
        sd = torch.where(empty, torch.ones_like(sd), sd)
        if not self.normalize:
            mean = torch.zeros_like(mean)
            sd = torch.ones_like(sd)
        normalized = torch.where(valid, (x-mean)/sd, torch.zeros_like(x))
        return normalized, mean, sd

    def _make_schedule(self, mask):
        """Generate original midpoint pivots from an unchanging (T,D) CPU mask."""
        T,D = mask.shape
        available = mask.copy()
        rounds = []
        max_rounds = int(math.ceil(math.log2(T)))+2
        for _ in range(max_rounds):
            if available.all():
                break
            entries = []
            for d in range(D):
                missing = np.flatnonzero(~available[:,d])
                if missing.size == 0:
                    continue
                splits = np.flatnonzero(np.diff(missing) != 1)+1
                for run in np.split(missing, splits):
                    a,z = int(run[0]),int(run[-1])
                    if a == 0 and z == T-1:
                        l,r = 0,T-1
                    else:
                        l,r = max(0,a-1),min(T-1,z+1)
                        if a == 0: l=r
                        if z == T-1: r=l
                    pivot=(a+z)//2
                    dist=max(1,r-l)
                    step=min(self.highest, 2**min(int(math.log2(dist)),int(math.log2(self.highest))))
                    entries.append((pivot,d,l,r,step,dist/T,(pivot-l)/max(1,dist)))
                    available[pivot,d] = True
            if not entries:
                raise RuntimeError('No missing pivots generated')
            rounds.append(entries)
        if not available.all():
            raise RuntimeError('Failed to fill all gaps')
        return rounds

    def forward(self, x_in, m_hat):
        if x_in.ndim != 3 or x_in.shape != m_hat.shape or x_in.shape[-1] != self.data_dim:
            raise ValueError('Inputs must be (B,T,D) and have matching masks')
        if not torch.isfinite(x_in).all().item():
            raise ValueError('X_in must be finite and prefilled')
        x,mean,sd = self._norm(x_in,m_hat)
        B,T,D = x.shape
        device = x.device
        mask_np = m_hat.detach().to('cpu').numpy() > 0.5   # once per forward; no per-pivot GPU sync
        per_sample = []
        for b in range(B):
            key = (T,D, np.packbits(mask_np[b].reshape(-1)).tobytes())
            schedule = self._schedule_cache.get(key)
            if schedule is None:
                schedule = self._make_schedule(mask_np[b])
                self._schedule_cache[key] = schedule
            per_sample.append(schedule)
        max_rounds = max((len(s) for s in per_sample),default=0)
        work = x
        available = m_hat.bool().clone()
        for round_i in range(max_rounds):
            recurrent,_ = self.encoder(torch.cat([work, available.to(work.dtype)],dim=-1))
            fwd = recurrent[:,:,:self.hidden_dim]
            bwd = recurrent[:,:,self.hidden_dim:]
            groups = {step:[] for step in (2**p for p in range(int(math.log2(self.highest))+1))}
            for b,schedule in enumerate(per_sample):
                if round_i >= len(schedule):
                    continue
                for t,d,l,r,step,dist_fraction,pivot_fraction in schedule[round_i]:
                    groups[step].append((b,t,d,l,r,dist_fraction,pivot_fraction))
            positions_list = []
            values_list = []
            for step, entries in groups.items():
                if not entries:
                    continue
                info_np = np.asarray(entries,dtype=np.float64)
                b = torch.as_tensor(info_np[:,0],device=device,dtype=torch.long)
                t = torch.as_tensor(info_np[:,1],device=device,dtype=torch.long)
                d = torch.as_tensor(info_np[:,2],device=device,dtype=torch.long)
                l = torch.as_tensor(info_np[:,3],device=device,dtype=torch.long)
                r = torch.as_tensor(info_np[:,4],device=device,dtype=torch.long)
                positions = b*(T*D)+t*D+d
                features = torch.cat((fwd[b,l],bwd[b,r],work[b,l],work[b,r],
                                      torch.as_tensor(info_np[:,5:7],device=device,dtype=work.dtype)),dim=-1)
                values = self.decoders[str(step)](features).gather(1,d[:,None]).squeeze(1)
                positions_list.append(positions)
                values_list.append(values)
            if not positions_list:
                raise RuntimeError('No missing pivots generated')
            positions=torch.cat(positions_list)
            values=torch.cat(values_list)
            work=work.reshape(-1).scatter(0,positions,values).view(B,T,D)
            available=available.reshape(-1).scatter(0,positions,
                        torch.ones_like(positions,dtype=torch.bool)).view(B,T,D)
        x_hat=work*sd+mean
        x_c=torch.where(m_hat.bool(),x_in,x_hat)
        return {'x_hat':x_hat,'x_c':x_c,'x_norm_hat':work,'mu':mean,'sigma':sd}


def masked_target_loss(outputs,truth,observed,holdout,loss_type='mse'):
    mask=holdout.bool() & (~observed.bool()) & torch.isfinite(truth)
    if not mask.any().item():
        return None
    errors=outputs['x_c'][mask]-truth[mask]
    if loss_type=='mse': return errors.square().mean()
    if loss_type=='mae': return errors.abs().mean()
    raise ValueError(loss_type)


### Model training

In [ ]:
NPZ_PATH='/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
OUT_DIR='/content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64'

In [ ]:
"""Patient-level 5-fold evaluation of CHANNEL-AWARE NAOMI ADAPTATION.

Run after placing naomi_channel_aware.py next to this script.
Not a reproduction of the unmodified official NAOMI full-sequence teacher-forcing
objective: this version uses normalized-input NAOMI-inspired recurrent
coarse-to-fine architecture, individual channel masks, and target-mask MSE.
Preserves original benchmark held-out I masks for train/val/test.
"""
import os,random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset,DataLoader


CKPT_DIR=os.path.join(OUT_DIR,'ckpts')
SEED=1337
N_SPLITS=5
VAL_WITHIN_TRAIN_P=.10
BATCH_SIZE= 64            
EPOCHS=300
LR=1e-3
WEIGHT_DECAY=0.
EARLY_STOPPING_PATIENCE=15
CLIP_GRAD_NORM=1.
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODEL_KWARGS=dict(data_dim=2,hidden_dim=64,decoder_hidden=128,
                  highest=16,rnn_layers=1,normalize=True)
MODEL_NAME='NAOMI_ChannelAware_Adapted'
REGIONS=['<1 min','[1,10) min','[10,20) min','[20,30] min']
EPS=1e-8

def seed_all(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)

def save(df,name):
    path=os.path.join(OUT_DIR,name);df.to_csv(path,index=False);return path

class DatasetImpute(Dataset):
    def __init__(self,arrays,indices):
        x,m,y,i=arrays
        self.x=torch.from_numpy(np.nan_to_num(x[indices],nan=0.,posinf=0.,neginf=0.).astype('float32'))
        self.m=torch.from_numpy(m[indices].astype('float32'))
        # Keep original non-finite target values; loss/evaluation masks exclude them.
        self.y=torch.from_numpy(y[indices].astype('float32'))
        self.i=torch.from_numpy(i[indices].astype('float32'))
        self.ids=indices
    def __len__(self):return len(self.ids)
    def __getitem__(self,j):return self.x[j],self.m[j],self.y[j],self.i[j],int(self.ids[j])

def make_loader(arrays,ids,shuffle=False):
    return DataLoader(DatasetImpute(arrays,ids),batch_size=BATCH_SIZE,shuffle=shuffle,
                      num_workers=0,pin_memory=DEVICE=='cuda')

def to_device(batch):
    x,m,y,i,ids=batch
    return x.to(DEVICE),m.to(DEVICE),y.to(DEVICE),i.to(DEVICE),ids

def gap_lengths(mask):
    m=np.asarray(mask)>0.5
    edges=np.diff(np.r_[False,m,False].astype('int8'))
    out=np.zeros(len(m),dtype='int32')
    for a,z in zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)):
        out[a:z]=z-a
    return out

def region(n):
    if 1<=n<12:return REGIONS[0]
    if 12<=n<120:return REGIONS[1]
    if 120<=n<240:return REGIONS[2]
    if 240<=n<=360:return REGIONS[3]
    return None

def metrics(frame):
    yt=frame.y_true.to_numpy(float);yp=frame.y_pred.to_numpy(float)
    valid=np.isfinite(yt)&np.isfinite(yp)
    yt=yt[valid];yp=yp[valid]
    if len(yt)==0:return dict(mae=np.nan,rmse=np.nan,mape_pct=np.nan,n_points=0)
    err=np.abs(yp-yt);nz=np.abs(yt)>EPS
    return dict(mae=float(err.mean()),rmse=float(np.sqrt((err**2).mean())),
                mape_pct=float(100*np.mean(err[nz]/np.abs(yt[nz]))) if nz.any() else np.nan,
                n_points=len(yt))

def run_train_epoch(model,loader,opt):
    model.train()
    total=0.;count=0
    for batch in loader:
        x,m,y,i,_=to_device(batch)
        opt.zero_grad(set_to_none=True)
        output=model(x,m)
        loss=masked_target_loss(output,y,m,i,loss_type='mse')
        if loss is None:continue
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Nonfinite NAOMI training MSE')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP_GRAD_NORM)
        opt.step()
        n=int((i.bool() & ~m.bool() & torch.isfinite(y)).sum())
        total+=float(loss.detach())*n;count+=n
    if count==0:raise ValueError('Training split has no valid artificial targets')
    return total/count

@torch.no_grad()
def run_eval(model,loader):
    model.eval();abs_sum=sq_sum=0.;n=0
    for batch in loader:
        x,m,y,i,_=to_device(batch)
        o=model(x,m)
        mask=i.bool()&(~m.bool())&torch.isfinite(y)
        error=o['x_c'][mask]-y[mask]
        if error.numel():
            abs_sum+=float(error.abs().sum());sq_sum+=float(error.square().sum())
            n+=error.numel()
    if not n:raise ValueError('No valid target points in validation/test split')
    return dict(mae=abs_sum/n,mse=sq_sum/n,rmse=(sq_sum/n)**.5,n_points=n)

@torch.no_grad()
def collect_predictions(model,loader,seg_id,filenames,fold,split):
    model.eval();rows=[]
    for batch in loader:
        x,m,y,i,indices=to_device(batch)
        preds=model(x,m)['x_c'].cpu().numpy()
        truth=y.cpu().numpy();target=i.cpu().numpy();visible=m.cpu().numpy()
        for b,idx in enumerate(indices.tolist()):
            for d,name in enumerate(['ScO2','MAP']):
                lengths=gap_lengths(target[b,:,d])
                for t in np.flatnonzero((target[b,:,d]>.5)&(visible[b,:,d]<=.5)):
                    yt=float(truth[b,t,d]);yp=float(preds[b,t,d])
                    if not (np.isfinite(yt) and np.isfinite(yp)):continue
                    g=int(lengths[t])
                    rows.append(dict(fold=fold,split=split,segment_idx=idx,
                                     patient_id=seg_id[idx],filename=str(filenames[idx]),
                                     time_idx=int(t),target=name,model=MODEL_NAME,
                                     y_true=yt,y_pred=yp,abs_err=abs(yp-yt),
                                     gap_len=g,gap_region=region(g)))
    return pd.DataFrame(rows)

def grouped_metrics(df,keys):
    if df.empty:return pd.DataFrame()
    results=[]
    for key,group in df.groupby(keys,observed=True):
        key=key if isinstance(key,tuple) else (key,)
        results.append({**dict(zip(keys,key)),**metrics(group)})
    return pd.DataFrame(results)

def save_summaries(split_rows,fold_rows,test_frames,val_frames):
    splits=pd.DataFrame(split_rows);folds=pd.DataFrame(fold_rows)
    tests=pd.concat(test_frames,ignore_index=True);vals=pd.concat(val_frames,ignore_index=True)
    save(splits,'cv5_patient_split_summary.csv')
    save(folds,'cv5_fold_results.csv')
    save(tests,'cv5_test_predictions_pooled.csv')
    save(vals,'cv5_validation_predictions_pooled.csv')
    for split,frame in [('test',tests),('validation',vals)]:
        gap=frame[frame.gap_region.notna()].copy()
        save(gap,f'cv5_{split}_predictions_pooled_with_gap_region.csv')
        save(grouped_metrics(frame,['target','model']),f'cv5_pooled_{split}_metrics.csv')
        save(grouped_metrics(gap,['target','model','gap_region']),f'cv5_pooled_{split}_metrics_by_gap_region.csv')
        fg=grouped_metrics(gap,['fold','target','model','gap_region'])
        save(fg,f'cv5_{split}_fold_metrics_by_gap_region.csv')
        if split=='test':save(fg,'cv5_fold_metrics_by_gap_region.csv')
        if not fg.empty:
            summ=fg.groupby(['target','model','gap_region'],observed=True,as_index=False).agg(
                mae_mean=('mae','mean'),mae_std=('mae','std'),
                rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
                mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),
                n_folds=('fold','nunique'),total_points=('n_points','sum'))
            save(summ,f'cv5_{split}_summary_mean_std_by_gap_region.csv')
            if split=='test':save(summ,'cv5_summary_mean_std_by_gap_region.csv')
    summ=folds.groupby(['target','model'],as_index=False).agg(
        val_mae_mean=('val_mae','mean'),val_mae_std=('val_mae','std'),
        test_mae_mean=('test_mae','mean'),test_mae_std=('test_mae','std'),
        test_rmse_mean=('test_rmse','mean'),test_rmse_std=('test_rmse','std'),
        test_mape_mean_pct=('test_mape_pct','mean'),test_mape_std_pct=('test_mape_pct','std'),
        n_folds=('fold','nunique'),total_points=('n_points','sum'))
    save(summ,'cv5_summary_mean_std.csv')
    print('\nFold summary:\n',summ.to_string(index=False))

def main():
    seed_all(SEED)
    os.makedirs(CKPT_DIR,exist_ok=True)
    with np.load(NPZ_PATH,allow_pickle=True) as d:
        x=np.asarray(d['X_in']);m=np.asarray(d['M_hat']);y=np.asarray(d['X']);i=np.asarray(d['I'])
        seg_id=np.asarray(d['segment_ids'])
        files=np.asarray(d['filenames']).astype(str) if 'filenames' in d.files else np.array([f'segment_{j}' for j in range(len(x))])
    if not (x.shape==m.shape==y.shape==i.shape) or x.shape[1:]!=(1080,2):
        raise ValueError('Expected aligned NPZ arrays (N,1080,2)')
    if np.any((m>.5)&(i>.5)):
        raise ValueError('M_hat and artificial I overlap')
    if not np.isfinite(y[i>.5]).all():raise ValueError('Nonfinite artificial target ground truth')
    patients=np.unique(seg_id)
    shuffled=patients.copy();np.random.default_rng(SEED).shuffle(shuffled)
    folds=np.array_split(shuffled,N_SPLITS)
    print('Dataset',x.shape,'patients',len(patients),'device',DEVICE)
    print('Channel ordering: ScO2, MAP. Training objective: I-mask MSE (adapted NAOMI)')
    arrays=(x,m,y,i)
    split_rows=[];fold_rows=[];test_frames=[];val_frames=[]
    for fold,test_pats in enumerate(folds,1):
        rest=patients[~np.isin(patients,test_pats)].copy()
        np.random.default_rng(SEED+fold).shuffle(rest)
        nv=min(max(1,round(len(rest)*VAL_WITHIN_TRAIN_P)),len(rest)-1)
        val_pats,train_pats=rest[:nv],rest[nv:]
        tr=np.flatnonzero(np.isin(seg_id,train_pats));va=np.flatnonzero(np.isin(seg_id,val_pats));te=np.flatnonzero(np.isin(seg_id,test_pats))
        print(f'\nFold {fold}/{N_SPLITS} segments train/val/test {len(tr)}/{len(va)}/{len(te)}')
        split_rows.append(dict(fold=fold,train_patients=','.join(map(str,train_pats)),
                 val_patients=','.join(map(str,val_pats)),test_patients=','.join(map(str,test_pats)),
                 n_train_patients=len(train_pats),n_val_patients=len(val_pats),n_test_patients=len(test_pats),
                 n_train_segments=len(tr),n_val_segments=len(va),n_test_segments=len(te)))
        # Persist splits before training so interrupted runs retain patient IDs.
        save(pd.DataFrame(split_rows),'cv5_patient_split_summary.csv')
        train_loader=make_loader(arrays,tr,True);val_loader=make_loader(arrays,va);test_loader=make_loader(arrays,te)
        seed_all(SEED+fold)
        model=ChannelAwareNAOMI(**MODEL_KWARGS).to(DEVICE)
        opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=3)
        ckpt=os.path.join(CKPT_DIR,f'fold{fold:02d}_best.pt')
        best=float('inf');best_ep=None;no_improve=0;history=[]
        for ep in range(1,EPOCHS+1):
            train_mse=run_train_epoch(model,train_loader,opt)
            val=run_eval(model,val_loader)
            scheduler.step(val['mae'])
            history.append(dict(fold=fold,epoch=ep,learning_rate=opt.param_groups[0]['lr'],
                                train_masked_mse=train_mse,val_masked_mse=val['mse'],
                                val_masked_mae=val['mae'],val_masked_rmse=val['rmse']))
            save(pd.DataFrame(history),f'fold{fold:02d}_training_history.csv')
            print(f'Epoch {ep:03d} lr={opt.param_groups[0]["lr"]:.2e} | train MSE={train_mse:.5f} | val MAE={val["mae"]:.5f} RMSE={val["rmse"]:.5f}')
            if val['mae']<best:
                best=val['mae'];best_ep=ep;no_improve=0
                torch.save(model.state_dict(),ckpt)
            else:
                no_improve+=1
                if no_improve>=EARLY_STOPPING_PATIENCE:
                    print('Early stop',ep);break
        model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
        vm=run_eval(model,val_loader)
        vp=collect_predictions(model,val_loader,seg_id,files,fold,'val')
        tp=collect_predictions(model,test_loader,seg_id,files,fold,'test')
        if vp.empty or tp.empty:raise ValueError('Empty prediction output')
        save(vp,f'fold{fold:02d}_validation_predictions.csv');save(tp,f'fold{fold:02d}_test_predictions.csv')
        val_frames.append(vp);test_frames.append(tp)
        val_by_target={k:metrics(frame) for k,frame in vp.groupby('target')}
        for channel,frame in tp.groupby('target'):
            tm=metrics(frame)
            fold_rows.append(dict(fold=fold,target=channel,model=MODEL_NAME,best_epoch=best_ep,
                                 val_mae=val_by_target[channel]['mae'],val_overall_mae=vm['mae'],
                                 test_mae=tm['mae'],test_rmse=tm['rmse'],test_mape_pct=tm['mape_pct'],
                                 n_points=tm['n_points']))
            print(f' TEST {channel}: MAE={tm["mae"]:.4f} RMSE={tm["rmse"]:.4f} MAPE={tm["mape_pct"]:.2f}%')
        save_summaries(split_rows,fold_rows,test_frames,val_frames)
        del model,opt,scheduler
        if DEVICE=='cuda':torch.cuda.empty_cache()
    print('Saved:',OUT_DIR)

if __name__=='__main__':main()

Dataset (3817, 1080, 2) patients 64 device cuda
Channel ordering: ScO2, MAP. Training objective: I-mask MSE (adapted NAOMI)

Fold 1/5 segments train/val/test 2814/256/747
Epoch 001 lr=1.00e-03 | train MSE=19.36818 | val MAE=2.09406 RMSE=3.28336
Epoch 002 lr=1.00e-03 | train MSE=9.73278 | val MAE=1.68949 RMSE=2.72061
Epoch 003 lr=1.00e-03 | train MSE=8.72280 | val MAE=1.77089 RMSE=2.82379
Epoch 004 lr=1.00e-03 | train MSE=7.99288 | val MAE=1.58955 RMSE=2.66742
Epoch 005 lr=1.00e-03 | train MSE=7.49157 | val MAE=1.62519 RMSE=2.66971
Epoch 006 lr=1.00e-03 | train MSE=7.65919 | val MAE=1.57140 RMSE=2.65254
Epoch 007 lr=1.00e-03 | train MSE=7.00569 | val MAE=1.61082 RMSE=2.69520
Epoch 008 lr=1.00e-03 | train MSE=7.17035 | val MAE=1.67577 RMSE=2.77638
Epoch 009 lr=1.00e-03 | train MSE=6.89251 | val MAE=1.60482 RMSE=2.64273
Epoch 010 lr=5.00e-04 | train MSE=6.89373 | val MAE=1.86611 RMSE=2.97314
Epoch 011 lr=5.00e-04 | train MSE=6.89148 | val MAE=1.69937 RMSE=2.77321
Epoch 012 lr=5.00e-04 | t

### Model Validation

In [ ]:
"""Recompute five-fold VALIDATION metrics from saved channel-aware NAOMI checkpoints.

Uses the exact ChannelAwareNAOMI architecture and patient splits saved by
train_naomi_channel_aware_5fold.py. NO retraining or optimizer updates.

The model is a channel-specific NAOMI-inspired adaptation, NOT the authors'
unmodified official NAOMI. ScO2 is channel 0; MAP is channel 1.
"""
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
SPLIT_CSV = os.path.join(OUT_DIR, 'cv5_patient_split_summary.csv')
N_SPLITS = 5
BATCH_SIZE = 8
EPS = 1e-8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'NAOMI_ChannelAware_Adapted'
MODEL_KWARGS = dict(data_dim=2,hidden_dim=64,decoder_hidden=128,
                    highest=16,rnn_layers=1,normalize=True)
CHANNELS = ['ScO2', 'MAP']
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']

class ValidationDataset(Dataset):
    def __init__(self, x, m, y, i, indices):
        self.x = torch.from_numpy(np.nan_to_num(x[indices],nan=0.,posinf=0.,neginf=0.).astype(np.float32))
        self.m = torch.from_numpy(m[indices].astype(np.float32))
        self.y = torch.from_numpy(y[indices].astype(np.float32))  # preserve missing GT
        self.i = torch.from_numpy(i[indices].astype(np.float32))
        self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        return self.x[idx], self.m[idx], self.y[idx], self.i[idx], int(self.indices[idx])

def parse_patients(value):
    if pd.isna(value) or not str(value).strip(): return []
    return [int(float(s.strip())) for s in str(value).split(',') if s.strip()]

def gap_lengths(mask):
    m = np.asarray(mask) > .5
    diff = np.diff(np.r_[False,m,False].astype(np.int8))
    result = np.zeros(len(m), dtype=np.int32)
    for a,z in zip(np.flatnonzero(diff==1), np.flatnonzero(diff==-1)):
        result[a:z] = z-a
    return result

def gap_region(g):
    if 1 <= g < 12: return REGIONS[0]
    if 12 <= g < 120: return REGIONS[1]
    if 120 <= g < 240: return REGIONS[2]
    if 240 <= g <= 360: return REGIONS[3]
    return None

def metrics(df):
    yt = df['y_true'].to_numpy(np.float64)
    yp = df['y_pred'].to_numpy(np.float64)
    good = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[good], yp[good]
    if len(yt) == 0:
        return dict(mae=np.nan,rmse=np.nan,mape_pct=np.nan,n_points=0,n_mape=0)
    ae = np.abs(yp-yt)
    nz = np.abs(yt) > EPS
    return dict(mae=float(ae.mean()),rmse=float(np.sqrt((ae**2).mean())),
                mape_pct=float(100*np.mean(ae[nz]/np.abs(yt[nz]))) if nz.any() else np.nan,
                n_points=len(yt),n_mape=int(nz.sum()))

def grouped(df, columns):
    rows=[]
    for names, part in df.groupby(columns, observed=True):
        if not isinstance(names, tuple):names=(names,)
        rows.append({**dict(zip(columns,names)),**metrics(part)})
    return pd.DataFrame(rows)

def save(df, filename):
    path=os.path.join(OUT_DIR,filename)
    df.to_csv(path,index=False)
    print('Saved:',path)

@torch.no_grad()
def run_fold(model,loader,patient_ids,filenames,fold):
    model.eval()
    abs_sum=sq_sum=0.
    n=0
    rows=[]
    for xb,mb,yb,ib,idxb in loader:
        xb=xb.to(DEVICE,non_blocking=True)
        mb=mb.to(DEVICE,non_blocking=True)
        # Ground truth never enters the model.
        prediction=model(xb,mb)['x_c'].detach().cpu().numpy()
        yt=yb.numpy(); obs=mb.cpu().numpy();hold=ib.numpy()
        for b,idx in enumerate(idxb.tolist()):
            for d,channel in enumerate(CHANNELS):
                lengths=gap_lengths(hold[b,:,d])
                target_times=np.flatnonzero((hold[b,:,d]>.5)&(obs[b,:,d]<=.5))
                for t in target_times:
                    actual=float(yt[b,t,d]); pred=float(prediction[b,t,d])
                    if not (np.isfinite(actual) and np.isfinite(pred)):
                        continue
                    error=pred-actual
                    abs_sum+=abs(error);sq_sum+=error**2;n+=1
                    g=int(lengths[t])
                    rows.append(dict(fold=fold,split='val',segment_idx=int(idx),
                                     patient_id=int(patient_ids[idx]),filename=str(filenames[idx]),
                                     time_idx=int(t),target=channel,model=MODEL_NAME,
                                     y_true=actual,y_pred=pred,abs_err=abs(error),
                                     gap_len=g,gap_region=gap_region(g)))
    if not n:raise ValueError(f'Fold {fold}: no finite evaluation targets')
    return dict(fold=fold,model=MODEL_NAME,val_imputation_mae=abs_sum/n,
                val_imputation_mse=sq_sum/n,val_imputation_rmse=np.sqrt(sq_sum/n),
                n_target_points=n),pd.DataFrame(rows)

def main():
    if not os.path.isfile(SPLIT_CSV):
        raise FileNotFoundError(f'Training patient split CSV not found: {SPLIT_CSV}')
    with np.load(NPZ_PATH,allow_pickle=True) as d:
        x=np.asarray(d['X_in']);m=np.asarray(d['M_hat']);y=np.asarray(d['X']);i=np.asarray(d['I'])
        patient_ids=np.asarray(d['segment_ids'])
        filenames=np.asarray(d['filenames']).astype(str) if 'filenames' in d.files else np.array([f'segment_{j}' for j in range(len(x))])
    if not(x.shape==m.shape==y.shape==i.shape) or x.shape[1:]!=(1080,2):
        raise ValueError('Expected aligned (N,1080,2) NPZ inputs')
    if len(patient_ids)!=len(x):raise ValueError('Patient IDs do not match segments')
    if np.any((m>.5)&(i>.5)):raise ValueError('Observed and artificial target masks overlap')
    if not np.isfinite(y[i>.5]).all():raise ValueError('Nonfinite artificial target ground truth')
    splits=pd.read_csv(SPLIT_CSV).sort_values('fold')
    if len(splits)!=N_SPLITS or set(splits.fold.astype(int))!=set(range(1,N_SPLITS+1)):
        raise ValueError('Expected exactly five completed saved patient splits')
    print('Device:',DEVICE,'| Model:',MODEL_NAME,'| Dataset:',x.shape)
    print('Evaluation uses channel-specific masks, saved best checkpoints, and artificial I targets.')
    print('No retraining. Validation loss is imputation MSE (not original NAOMI teacher-forcing loss).')
    fold_losses=[];all_preds=[]
    for _,r in splits.iterrows():
        fold=int(r['fold'])
        train_pats=set(parse_patients(r['train_patients']))
        val_pats=set(parse_patients(r['val_patients']))
        test_pats=set(parse_patients(r['test_patients']))
        if not val_pats or train_pats&val_pats or val_pats&test_pats or train_pats&test_pats:
            raise ValueError(f'Fold {fold}: empty validation set or overlapping patient partitions')
        indices=np.flatnonzero(np.isin(patient_ids,list(val_pats)))
        if len(indices)==0:raise ValueError(f'Fold {fold}: no validation segments')
        if 'n_val_segments' in r.index and pd.notna(r['n_val_segments']) and len(indices)!=int(r['n_val_segments']):
            raise ValueError(f'Fold {fold}: validation segments do not match saved split')
        loader=DataLoader(ValidationDataset(x,m,y,i,indices),batch_size=BATCH_SIZE,
                          shuffle=False,num_workers=0,pin_memory=DEVICE.type=='cuda')
        ckpt=os.path.join(CKPT_DIR,f'fold{fold:02d}_best.pt')
        if not os.path.isfile(ckpt):raise FileNotFoundError(ckpt)
        model=ChannelAwareNAOMI(**MODEL_KWARGS).to(DEVICE)
        model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True),strict=True)
        score,preds=run_fold(model,loader,patient_ids,filenames,fold)
        score.update(n_val_patients=len(val_pats),n_val_segments=len(indices))
        fold_losses.append(score);all_preds.append(preds)
        save(preds,f'fold{fold:02d}_validation_predictions_recomputed.csv')
        print(f"Fold {fold}: validation MSE={score['val_imputation_mse']:.6f}, "
              f"MAE={score['val_imputation_mae']:.6f}, RMSE={score['val_imputation_rmse']:.6f}, "
              f"points={score['n_target_points']}")
        for channel,part in preds.groupby('target'):
            q=metrics(part)
            print(f"  {channel}: MAE={q['mae']:.5f}, RMSE={q['rmse']:.5f}, MAPE={q['mape_pct']:.3f}%")
        del model
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    losses=pd.DataFrame(fold_losses)
    save(losses,'cv5_validation_losses_recomputed.csv')
    overall=pd.DataFrame([dict(model=MODEL_NAME,
        val_imputation_mae_mean=losses.val_imputation_mae.mean(),val_imputation_mae_std=losses.val_imputation_mae.std(ddof=1),
        val_imputation_mse_mean=losses.val_imputation_mse.mean(),val_imputation_mse_std=losses.val_imputation_mse.std(ddof=1),
        val_imputation_rmse_mean=losses.val_imputation_rmse.mean(),val_imputation_rmse_std=losses.val_imputation_rmse.std(ddof=1),
        n_folds=len(losses))])
    save(overall,'cv5_validation_loss_summary_mean_std_recomputed.csv')
    pooled=pd.concat(all_preds,ignore_index=True)
    save(pooled,'cv5_validation_predictions_recomputed_pooled.csv')
    gap=pooled[pooled.gap_region.notna()].copy()
    gap['gap_region']=pd.Categorical(gap['gap_region'],REGIONS,ordered=True)
    save(gap,'cv5_validation_predictions_recomputed_with_gap_region.csv')
    fold_metrics=grouped(pooled,['fold','target','model'])
    fold_gap=grouped(gap,['fold','target','model','gap_region'])
    save(fold_metrics,'cv5_validation_fold_metrics_recomputed.csv')
    save(fold_gap,'cv5_validation_fold_metrics_recomputed_by_gap_region.csv')
    save(grouped(pooled,['target','model']),'cv5_validation_pooled_metrics_recomputed.csv')
    save(grouped(gap,['target','model','gap_region']),'cv5_validation_pooled_metrics_recomputed_by_gap_region.csv')
    per_channel=fold_metrics.groupby(['target','model'],as_index=False).agg(
        mae_mean=('mae','mean'),mae_std=('mae','std'),rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
        mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),
        n_folds=('fold','nunique'),total_points=('n_points','sum'))
    save(per_channel,'cv5_validation_metrics_recomputed_mean_std.csv')
    gap_summary=fold_gap.groupby(['target','model','gap_region'],observed=True,as_index=False).agg(
        mae_mean=('mae','mean'),mae_std=('mae','std'),rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
        mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),
        n_folds=('fold','nunique'),total_points=('n_points','sum'))
    save(gap_summary,'cv5_validation_metrics_recomputed_mean_std_by_gap_region.csv')
    print('\nFIVE-FOLD VALIDATION OVERALL MEAN ± SD:\n',overall.to_string(index=False))
    print('\nVALIDATION MAE/RMSE/MAPE BY CHANNEL:\n',per_channel.to_string(index=False))
    print('\nVALIDATION METRICS BY GAP LENGTH:\n',gap_summary.to_string(index=False))
    print('\nSaved all results under:',OUT_DIR)

if __name__=='__main__':main()


Device: cuda | Model: NAOMI_ChannelAware_Adapted | Dataset: (3817, 1080, 2)
Evaluation uses channel-specific masks, saved best checkpoints, and artificial I targets.
No retraining. Validation loss is imputation MSE (not original NAOMI teacher-forcing loss).
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/fold01_validation_predictions_recomputed.csv
Fold 1: validation MSE=6.279629, MAE=1.484323, RMSE=2.505919, points=9917
  MAP: MAE=1.36352, RMSE=2.23174, MAPE=2.021%
  ScO2: MAE=1.51866, RMSE=2.57854, MAPE=2.508%
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/fold02_validation_predictions_recomputed.csv
Fold 2: validation MSE=3.478902, MAE=1.135185, RMSE=1.865182, points=13337
  MAP: MAE=1.07420, RMSE=1.98502, MAPE=1.784%
  ScO2: MAE=1.15457, RMSE=1.82544, MAPE=1.737%
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/fold03_validation_predictions_recomputed.csv
Fold 3: validation MSE=8.093907, MAE=1.179534, RMSE=2.844979, points

### Model Testing

In [ ]:
"""Channel-aware NAOMI: five-fold held-out TEST MAE/MAPE by gap duration.

Reads cv5_test_predictions_pooled.csv generated by
train_naomi_channel_aware_5fold.py. No training or checkpoint loading.
Channel 0 = ScO2, channel 1 = MAP. Sampling interval = 5 seconds.
"""
import os
import numpy as np
import pandas as pd
MODEL_NAME = 'NAOMI_ChannelAware_Adapted'
EPS = 1e-8
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']


def calculate_metrics(sdf):
    yt = sdf['y_true'].to_numpy(dtype=np.float64)
    yp = sdf['y_pred'].to_numpy(dtype=np.float64)
    ae = np.abs(yp - yt)
    nonzero = np.abs(yt) > EPS
    return dict(
        mae=float(ae.mean()),
        mape_pct=float(100 * np.mean(ae[nonzero] / np.abs(yt[nonzero]))) if nonzero.any() else np.nan,
        n_points_total=int(len(yt)),
        n_points_used_mape=int(nonzero.sum()),
        n_points_excluded_gt0=int((~nonzero).sum()),
    )


def mean_sd(mean, sd):
    if pd.isna(mean):
        return 'NA'
    return f'{mean:.2f} ± {sd:.2f}' if pd.notna(sd) else f'{mean:.2f} ± NA'


def main(out_dir=OUT_DIR):
    path = os.path.join(out_dir, 'cv5_test_predictions_pooled.csv')
    if not os.path.isfile(path):
        raise FileNotFoundError(f'NAOMI test predictions not found: {path}\nRun five-fold training/testing first.')
    pred = pd.read_csv(path)
    required = {'fold','target','y_true','y_pred','gap_len'}
    missing = required - set(pred.columns)
    if missing:
        raise ValueError(f'Missing prediction columns: {sorted(missing)}')
    if 'split' in pred.columns and not pred['split'].astype(str).str.lower().eq('test').all():
        raise ValueError('Non-test predictions found in the test CSV')
    if 'model' not in pred.columns:
        pred['model'] = MODEL_NAME
    if set(pred['model'].dropna().unique()) != {MODEL_NAME}:
        raise ValueError(f'Expected only {MODEL_NAME}; found {pred["model"].unique().tolist()}')
    for name in ['fold','y_true','y_pred','gap_len']:
        pred[name] = pd.to_numeric(pred[name], errors='coerce')
    pred = pred.loc[np.isfinite(pred['fold']) & np.isfinite(pred['y_true']) & np.isfinite(pred['y_pred'])].copy()
    if pred.empty:
        raise ValueError('No finite held-out test predictions')
    pred['fold'] = pred['fold'].astype(int)
    folds = sorted(pred['fold'].unique().tolist())
    if folds != [1,2,3,4,5]:
        raise ValueError(f'Expected folds 1..5; found {folds}')

    mapping = {'channel_0':'ScO2','channel_1':'MAP','0':'ScO2','1':'MAP','ScO₂':'ScO2'}
    pred['target'] = pred['target'].astype(str).replace(mapping)
    if set(pred['target'].unique()) != {'ScO2','MAP'}:
        raise ValueError(f'Unexpected channel labels: {pred["target"].unique().tolist()}')

    pred['gap_region'] = pd.cut(pred['gap_len'],bins=[0,11,119,239,360],
                                labels=REGIONS[:-1],right=True,include_lowest=False)
    outside = int(pred['gap_region'].isna().sum())
    if outside:
        print(f'WARNING: {outside} points outside gap_len 1..360; included in ALL only')
    print('Model:',MODEL_NAME,'| Channel 0: ScO2; Channel 1: MAP')
    print('Held-out test points by gap category:\n',pred['gap_region'].value_counts(sort=False).to_string())
    print('ALL held-out test points:',len(pred))

    gap = pred.loc[pred['gap_region'].notna()].copy()
    all_pts = pred.copy()
    all_pts['gap_region'] = 'ALL'
    ev = pd.concat([gap,all_pts],ignore_index=True)
    ev['gap_region'] = pd.Categorical(ev['gap_region'],categories=REGIONS,ordered=True)

    fold_df = pd.DataFrame([
        {'fold':int(f),'target':target,'model':model,'gap_region':str(region),**calculate_metrics(sub)}
        for (f,target,model,region),sub in ev.groupby(['fold','target','model','gap_region'],observed=True)
    ])
    fold_df['gap_region'] = pd.Categorical(fold_df['gap_region'],categories=REGIONS,ordered=True)
    fold_df = fold_df.sort_values(['target','model','gap_region','fold']).reset_index(drop=True)

    summary = fold_df.groupby(['target','model','gap_region'],observed=True).agg(
        mae_mean=('mae','mean'),mae_std=('mae','std'),
        mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),
        n_folds_mae=('mae','count'),n_folds_mape=('mape_pct','count'),
        total_points=('n_points_total','sum'),
        total_points_used_mape=('n_points_used_mape','sum'),
        total_points_excluded_gt0=('n_points_excluded_gt0','sum')
    ).reset_index().sort_values(['target','model','gap_region']).reset_index(drop=True)

    pooled = pd.DataFrame([
        {'target':target,'model':model,'gap_region':str(region),**calculate_metrics(sub)}
        for (target,model,region),sub in ev.groupby(['target','model','gap_region'],observed=True)
    ])
    pooled['gap_region'] = pd.Categorical(pooled['gap_region'],categories=REGIONS,ordered=True)
    pooled = pooled.sort_values(['target','model','gap_region']).reset_index(drop=True)

    paper_rows = []
    for (target,model),sub in summary.groupby(['target','model'],observed=True):
        row = {'target':target,'model':model}
        for region in REGIONS:
            matching = sub.loc[sub['gap_region'] == region]
            if matching.empty:
                row[f'{region} | MAE'] = 'NA'
                row[f'{region} | MAPE(%)'] = 'NA'
            else:
                r = matching.iloc[0]
                row[f'{region} | MAE'] = mean_sd(r['mae_mean'],r['mae_std'])
                row[f'{region} | MAPE(%)'] = mean_sd(r['mape_mean_pct'],r['mape_std_pct'])
        paper_rows.append(row)
    paper = pd.DataFrame(paper_rows)

    os.makedirs(out_dir,exist_ok=True)
    outputs = {
        'cv5_fold_mae_mape_by_gap_region.csv':fold_df,
        'cv5_summary_mean_std_mae_mape_by_gap_region.csv':summary,
        'cv5_pooled_test_mae_mape_by_gap_region.csv':pooled,
        'naomi_paper4_mae_mape_table.csv':paper,
    }
    for filename,frame in outputs.items():
        dst = os.path.join(out_dir,filename)
        frame.to_csv(dst,index=False)
        print('Saved:',dst)
    print('\nFOLDWISE TEST MAE + MAPE:\n',fold_df.to_string(index=False))
    print('\nFIVE-FOLD MEAN ± SAMPLE SD:\n',summary.to_string(index=False))
    print('\nMANUSCRIPT-READY TABLE:\n',paper.to_string(index=False))
    return fold_df,summary,pooled,paper


if __name__ == '__main__':
    main()


Model: NAOMI_ChannelAware_Adapted | Channel 0: ScO2; Channel 1: MAP
Held-out test points by gap category:
 gap_region
<1 min         39503
[1,10) min     52673
[10,20) min    26458
[20,30] min    25794
ALL held-out test points: 144428
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/cv5_fold_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/cv5_summary_mean_std_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/cv5_pooled_test_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/naomi_paper4_mae_mape_table.csv

FOLDWISE TEST MAE + MAPE:
  fold target                      model  gap_region      mae  mape_pct  n_points_total  n_points_used_mape  n_points_excluded_gt0
    1    MAP NAOMI_ChannelAware_Adapted      <1 min 1.161045  4.435939            2570                2570                      0
    2    MAP NAOMI_ChannelAware_Adapted      <